# Chronos-2 ReNile-IOT Evaluation

This notebook evaluates Chronos-2 using only ReNile-IOT data.

- Fetch data from `2026-05-01 00:00` onward.
- Use `2026-05-01 00:00` through `2026-05-14 23:00` as the 336-hour context.
- Forecast `2026-05-15 00:00` through `2026-05-21 23:00` with a 168-hour horizon.
- Interpolation is used only to prepare the model context.
- Metrics are computed only against raw ReNile-IOT actual observations, never interpolated actuals.

## Imports

In [6]:
from __future__ import annotations

from getpass import getpass
import math
import os
from pathlib import Path
import sys
import time

import httpx
import numpy as np
import pandas as pd
import torch
from chronos import BaseChronosPipeline

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.core.config import get_settings
from src.core.logging import configure_logging
from src.services.weather.providers.renile_iot import parse_renile_iot_payload

## Configuration

Set `RENILE_IOT_JWT` and `RENILE_IOT_DEVICE_ID` in your environment, or enter them when prompted.

In [7]:
settings = get_settings()
configure_logging(settings.logging_level)

JWT = os.getenv("RENILE_IOT_JWT") or getpass("ReNile-IOT JWT: ")
DEVICE_ID = os.getenv("RENILE_IOT_DEVICE_ID") or input("ReNile-IOT device_id: " ).strip()

FETCH_START = pd.Timestamp("2026-05-01 00:00")
CONTEXT_START = pd.Timestamp("2026-05-01 00:00")
CONTEXT_END = pd.Timestamp("2026-05-14 23:00")
FORECAST_START = pd.Timestamp("2026-05-15 00:00")
FORECAST_END = pd.Timestamp("2026-05-21 23:00")
CONTEXT_HOURS = 336
PREDICTION_LENGTH = 168

MODEL_ID = settings.chronos_model_id
DEVICE_MAP = settings.chronos_device_map if torch.cuda.is_available() else "cpu"

print(f"Model: {MODEL_ID}")
print(f"Device: {DEVICE_MAP}")
print(f"Context: {CONTEXT_START} -> {CONTEXT_END} ({CONTEXT_HOURS} hours)")
print(f"Forecast: {FORECAST_START} -> {FORECAST_END} ({PREDICTION_LENGTH} hours)")

Model: amazon/chronos-2
Device: cuda
Context: 2026-05-01 00:00:00 -> 2026-05-14 23:00:00 (336 hours)
Forecast: 2026-05-15 00:00:00 -> 2026-05-21 23:00:00 (168 hours)


## Fetch Raw ReNile-IOT Data

In [8]:
def fetch_renile_iot_payload() -> dict:
    params = {
        "data_type": settings.renile_iot_data_type,
        "start_time": FETCH_START.strftime("%Y-%m-%d %H:%M"),
        "device_id": DEVICE_ID,
    }
    headers = {
        "Authorization": f"JWT {JWT}",
        "Accept": "application/json",
    }
    with httpx.Client(timeout=settings.renile_iot_timeout_seconds) as client:
        response = client.get(settings.renile_iot_url, params=params, headers=headers)
        response.raise_for_status()
    return response.json()


def payload_label_summary(payload: dict) -> pd.DataFrame:
    rows = []
    for sensor_name, sensor_data in payload.items():
        labels = sensor_data.get("labels", []) if isinstance(sensor_data, dict) else []
        parsed = pd.to_datetime(labels, utc=True, errors="coerce")
        rows.append(
            {
                "sensor": sensor_name,
                "label_count": len(labels),
                "first_raw_label": labels[0] if labels else None,
                "last_raw_label": labels[-1] if labels else None,
                "first_timestamp_utc": parsed.min(),
                "last_timestamp_utc": parsed.max(),
                "invalid_labels": int(parsed.isna().sum()),
            }
        )
    return pd.DataFrame(rows)


payload = fetch_renile_iot_payload()
label_summary_df = payload_label_summary(payload)
raw_renile_df = parse_renile_iot_payload(payload)
raw_renile_df["timestamp"] = pd.to_datetime(raw_renile_df["timestamp"], utc=True, errors="coerce").dt.tz_localize(None)
raw_renile_df = raw_renile_df.dropna(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)

raw_eval_span_df = raw_renile_df[
    (raw_renile_df["timestamp"] >= CONTEXT_START)
    & (raw_renile_df["timestamp"] <= FORECAST_END)
].copy()

print(f"Raw rows, unfiltered: {len(raw_renile_df)}")
print(f"Raw rows, context+forecast span: {len(raw_eval_span_df)}")
print(f"Sensors: {raw_renile_df.columns.drop('timestamp').tolist()}")
print(f"Unfiltered raw range: {raw_renile_df['timestamp'].min()} -> {raw_renile_df['timestamp'].max()}")
print(f"Evaluation span raw range: {raw_eval_span_df['timestamp'].min()} -> {raw_eval_span_df['timestamp'].max()}")
display(label_summary_df)
display(raw_eval_span_df.head())
display(raw_eval_span_df.tail())

2026-06-17 12:42:31,834 INFO [src.services.weather.providers.renile_iot] Parsing ReNile-IOT payload sensors=['SO2', 'NO2', 'CO', 'O3', 'PM2_5', 'PM10', 'ambient_temp', 'ambinet_Humi', 'Weather_Pressure', 'Wind_Speed', 'Wind_Direction', 'Rain_Fall']
2026-06-17 12:42:31,839 INFO [src.services.weather.providers.renile_iot] ReNile-IOT sensor parsed sensor=SO2 readings=293 valid_timestamps=293 start=2026-04-30 22:00:00 end=2026-06-17 08:00:00 preview={'head': [{'timestamp': Timestamp('2026-04-30 22:00:00'), 'SO2': 8.0}, {'timestamp': Timestamp('2026-04-30 23:00:00'), 'SO2': 9.0}, {'timestamp': Timestamp('2026-05-01 00:00:00'), 'SO2': 8.0}], 'tail': [{'timestamp': Timestamp('2026-06-16 20:00:00'), 'SO2': 20.0}, {'timestamp': Timestamp('2026-06-16 21:00:00'), 'SO2': 27.0}, {'timestamp': Timestamp('2026-06-17 08:00:00'), 'SO2': 63.0}]}
2026-06-17 12:42:31,842 INFO [src.services.weather.providers.renile_iot] ReNile-IOT sensor parsed sensor=NO2 readings=293 valid_timestamps=293 start=2026-04-30 

,sensor,label_count,first_raw_label,last_raw_label,first_timestamp_utc,last_timestamp_utc,invalid_labels
0,SO2,293,2026-04-30T22:00:00.000Z,2026-06-17T08:00:00.000Z,2026-04-30 22:00:00+00:00,2026-06-17 08:00:00+00:00,0
1,NO2,293,2026-04-30T22:00:00.000Z,2026-06-17T08:00:00.000Z,2026-04-30 22:00:00+00:00,2026-06-17 08:00:00+00:00,0
2,CO,293,2026-04-30T22:00:00.000Z,2026-06-17T08:00:00.000Z,2026-04-30 22:00:00+00:00,2026-06-17 08:00:00+00:00,0
3,O3,293,2026-04-30T22:00:00.000Z,2026-06-17T08:00:00.000Z,2026-04-30 22:00:00+00:00,2026-06-17 08:00:00+00:00,0
4,PM2_5,293,2026-04-30T22:00:00.000Z,2026-06-17T08:00:00.000Z,2026-04-30 22:00:00+00:00,2026-06-17 08:00:00+00:00,0
5,PM10,294,2026-04-30T22:00:00.000Z,2026-06-17T08:00:00.000Z,2026-04-30 22:00:00+00:00,2026-06-17 08:00:00+00:00,0
6,ambient_temp,294,2026-04-30T22:00:00.000Z,2026-06-17T08:00:00.000Z,2026-04-30 22:00:00+00:00,2026-06-17 08:00:00+00:00,0
7,ambinet_Humi,294,2026-04-30T22:00:00.000Z,2026-06-17T08:00:00.000Z,2026-04-30 22:00:00+00:00,2026-06-17 08:00:00+00:00,0
8,Weather_Pressure,294,2026-04-30T22:00:00.000Z,2026-06-17T08:00:00.000Z,2026-04-30 22:00:00+00:00,2026-06-17 08:00:00+00:00,0
9,Wind_Speed,294,2026-04-30T22:00:00.000Z,2026-06-17T08:00:00.000Z,2026-04-30 22:00:00+00:00,2026-06-17 08:00:00+00:00,0


,timestamp,SO2,NO2,CO,O3,PM2_5,PM10,ambient_temp,ambinet_Humi,Weather_Pressure,Wind_Speed,Wind_Direction,Rain_Fall
2,2026-05-01 00:00:00,8.0,16.0,35.0,30.0,30.0,34.0,20.32,60.11,1009.3,0.62,330.3,0.0
3,2026-05-07 08:00:00,9.0,9.0,43.0,34.0,32.0,35.0,24.91,41.24,1014.4,0.50,253.3,0.0
4,2026-05-07 09:00:00,8.0,9.0,41.0,40.0,31.0,33.0,25.64,38.47,1013.8,1.30,18.9,0.0
5,2026-05-07 10:00:00,9.0,9.0,46.0,39.0,25.0,25.0,27.44,33.77,1013.3,1.24,218.2,0.0
6,2026-05-07 11:00:00,9.0,9.0,82.0,40.0,25.0,25.0,28.95,32.03,1012.6,0.59,323.6,0.0


,timestamp,SO2,NO2,CO,O3,PM2_5,PM10,ambient_temp,ambinet_Humi,Weather_Pressure,Wind_Speed,Wind_Direction,Rain_Fall
11,2026-05-07 18:00:00,9.0,10.0,48.0,30.0,29.0,30.0,21.37,48.78,1013.7,0.96,254.3,0.0
12,2026-05-07 19:00:00,9.0,11.0,41.0,28.0,25.0,27.0,20.39,54.34,1014.7,0.52,323.1,0.0
13,2026-05-07 20:00:00,9.0,17.0,42.0,26.0,27.0,28.0,19.69,59.55,1014.8,0.09,29.3,0.0
14,2026-05-07 21:00:00,7.0,19.0,36.0,31.0,22.0,23.0,19.28,63.28,1014.6,0.95,312.8,0.0
15,2026-05-07 22:00:00,7.0,23.0,33.0,28.0,25.0,29.0,18.87,67.11,1014.6,1.15,335.5,0.0


## Build Model Context

The model context must be regular hourly data. Missing context values are interpolated here only. The forecast-period actuals remain raw and are not interpolated.

In [9]:
context_raw_df = raw_renile_df[
    (raw_renile_df["timestamp"] >= CONTEXT_START)
    & (raw_renile_df["timestamp"] <= CONTEXT_END)
].copy()
actual_raw_df = raw_renile_df[
    (raw_renile_df["timestamp"] >= FORECAST_START)
    & (raw_renile_df["timestamp"] <= FORECAST_END)
].copy()

sensor_columns = [column for column in raw_renile_df.columns if column != "timestamp"]
context_targets = [target for target in sensor_columns if context_raw_df[target].notna().any()]
actual_targets = [target for target in sensor_columns if actual_raw_df[target].notna().any()]
targets = [target for target in context_targets if target in actual_targets]

availability_df = pd.DataFrame(
    {
        "sensor": sensor_columns,
        "context_non_null": [int(context_raw_df[sensor].notna().sum()) for sensor in sensor_columns],
        "raw_forecast_non_null": [int(actual_raw_df[sensor].notna().sum()) for sensor in sensor_columns],
    }
)

print(f"Raw context rows: {len(context_raw_df)}")
print(f"Raw forecast rows: {len(actual_raw_df)}")
print(f"Context raw range: {context_raw_df['timestamp'].min()} -> {context_raw_df['timestamp'].max()}")
print(f"Forecast raw range: {actual_raw_df['timestamp'].min()} -> {actual_raw_df['timestamp'].max()}")
display(availability_df)

if actual_raw_df.empty:
    nearest_before = raw_renile_df.loc[raw_renile_df["timestamp"] < FORECAST_START, "timestamp"].max()
    nearest_after = raw_renile_df.loc[raw_renile_df["timestamp"] > FORECAST_END, "timestamp"].min()
    raise ValueError(
        "No raw ReNile-IOT rows exist in the forecast window. "
        f"Check API date filtering/data availability/device_id. "
        f"Nearest raw timestamp before window: {nearest_before}; after window: {nearest_after}."
    )
if not targets:
    raise ValueError("No sensors have both context data and raw actual forecast-period data.")

processed_context_df = (
    context_raw_df[["timestamp", *targets]]
    .drop_duplicates(subset=["timestamp"], keep="last")
    .sort_values("timestamp")
    .set_index("timestamp")
    .asfreq("h")
)
processed_context_df = processed_context_df.interpolate(method="time").ffill().bfill()
processed_context_df = processed_context_df.loc[CONTEXT_START:CONTEXT_END].reset_index()

if len(processed_context_df) != CONTEXT_HOURS:
    raise ValueError(f"Expected {CONTEXT_HOURS} context hours, got {len(processed_context_df)}.")
if processed_context_df[targets].isna().any().any():
    raise ValueError("Context still contains missing values after interpolation.")

chronos_context = pd.DataFrame({"item_id": "weather_series", "timestamp": processed_context_df["timestamp"]})
for target in targets:
    chronos_context[target] = processed_context_df[target].astype(float)

print(f"Targets used: {targets}")
print(f"Context rows: {len(chronos_context)}")
print(f"Raw actual rows before melt: {len(actual_raw_df)}")
display(chronos_context.head())
display(chronos_context.tail())

Raw context rows: 14
Raw forecast rows: 0
Context raw range: 2026-05-01 00:00:00 -> 2026-05-07 22:00:00
Forecast raw range: NaT -> NaT


,sensor,context_non_null,raw_forecast_non_null
0,SO2,14,0
1,NO2,14,0
2,CO,14,0
3,O3,14,0
4,PM2_5,14,0
5,PM10,14,0
6,ambient_temp,14,0
7,ambinet_Humi,14,0
8,Weather_Pressure,14,0
9,Wind_Speed,14,0


ValueError: No raw ReNile-IOT rows exist in the forecast window. Check API date filtering/data availability/device_id. Nearest raw timestamp before window: 2026-05-07 22:00:00; after window: 2026-05-23 08:00:00.

## Forecast

In [ ]:
load_start = time.perf_counter()
pipeline = BaseChronosPipeline.from_pretrained(MODEL_ID, device_map=DEVICE_MAP)
load_seconds = time.perf_counter() - load_start

forecast_start_time = time.perf_counter()
predictions_df = pipeline.predict_df(
    chronos_context,
    prediction_length=PREDICTION_LENGTH,
    quantile_levels=[0.1, 0.5, 0.9],
    timestamp_column="timestamp",
    target=targets,
)
inference_seconds = time.perf_counter() - forecast_start_time

predictions_df = predictions_df.rename(
    columns={"predictions": "prediction", "0.1": "q10", "0.5": "q50", "0.9": "q90", 0.1: "q10", 0.5: "q50", 0.9: "q90"}
)
predictions_df["timestamp"] = pd.to_datetime(predictions_df["timestamp"], utc=True).dt.tz_localize(None)
if "target_name" not in predictions_df.columns:
    if len(targets) != 1:
        raise ValueError("Chronos output is missing target_name for multi-target predictions.")
    predictions_df["target_name"] = targets[0]
predictions_df = predictions_df.sort_values(["target_name", "timestamp"]).reset_index(drop=True)

print(f"Model load seconds: {load_seconds:.2f}")
print(f"Inference seconds: {inference_seconds:.2f}")
print(f"Prediction rows: {len(predictions_df)}")
display(predictions_df.head())
display(predictions_df.tail())

## Compare With Raw ReNile-IOT Actuals

This section drops missing raw actuals and merges predictions only with timestamps that were actually returned by ReNile-IOT. No interpolated actual values are used.

In [ ]:
actual_long = actual_raw_df[["timestamp", *targets]].melt(
    id_vars=["timestamp"],
    var_name="target_name",
    value_name="actual",
).dropna(subset=["actual"])
actual_long["timestamp"] = pd.to_datetime(actual_long["timestamp"], utc=True).dt.tz_localize(None)
actual_long["actual"] = pd.to_numeric(actual_long["actual"], errors="coerce")
actual_long = actual_long.dropna(subset=["actual"])

comparison_df = actual_long.merge(
    predictions_df[["timestamp", "target_name", "prediction", "q10", "q50", "q90"]],
    on=["timestamp", "target_name"],
    how="inner",
)

expected_prediction_timestamps = pd.date_range(FORECAST_START, FORECAST_END, freq="h")
non_hourly_actuals = actual_long.loc[actual_long["timestamp"] != actual_long["timestamp"].dt.floor("h"), "timestamp"].drop_duplicates()
actual_timestamp_overlap = set(actual_long["timestamp"].drop_duplicates()).intersection(set(expected_prediction_timestamps))

if comparison_df.empty:
    print(f"Raw actual timestamp range: {actual_long['timestamp'].min()} -> {actual_long['timestamp'].max()}")
    print(f"Prediction timestamp range: {predictions_df['timestamp'].min()} -> {predictions_df['timestamp'].max()}")
    print(f"Exact hourly timestamp overlap before sensor join: {len(actual_timestamp_overlap)}")
    print(f"Non-hourly raw actual timestamps: {len(non_hourly_actuals)}")
    raise ValueError("No raw ReNile-IOT actual timestamps matched Chronos prediction timestamps after exact inner join.")

actual_matched_timestamps = comparison_df["timestamp"].nunique()
print(f"Raw actual values available for scoring: {len(actual_long)}")
print(f"Prediction/actual comparison rows: {len(comparison_df)}")
print(f"Matched forecast timestamps: {actual_matched_timestamps} / {len(expected_prediction_timestamps)}")
display(comparison_df.head())
display(comparison_df.tail())

## Metrics

For each target, `error = prediction - actual`.

- `mae`: mean absolute error.
- `mse`: mean squared error.
- `rmse`: square root of MSE.
- `bias`: mean error; positive means overprediction, negative means underprediction.
- `q10_q90_coverage_percent`: percent of raw actual values inside Chronos' 10%-90% interval.

In [ ]:
def safe_mape(actual: np.ndarray, predicted: np.ndarray) -> float:
    mask = np.abs(actual) > 1e-8
    if not mask.any():
        return math.nan
    return float(np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100)


def r2_score(actual: np.ndarray, predicted: np.ndarray) -> float:
    ss_res = float(np.sum((actual - predicted) ** 2))
    ss_tot = float(np.sum((actual - actual.mean()) ** 2))
    if ss_tot == 0:
        return math.nan
    return 1 - (ss_res / ss_tot)


def is_direction_target(target_name: str) -> bool:
    return "direction" in target_name.lower()


def angular_error_degrees(predicted: np.ndarray, actual: np.ndarray) -> np.ndarray:
    return ((predicted - actual + 180) % 360) - 180


rows = []
for target_name, group in comparison_df.groupby("target_name", sort=True):
    actual = group["actual"].to_numpy(dtype=float)
    predicted = group["prediction"].to_numpy(dtype=float)

    if is_direction_target(target_name):
        errors = angular_error_degrees(predicted, actual)
        mape = math.nan
        r2 = math.nan
    else:
        errors = predicted - actual
        mape = safe_mape(actual, predicted)
        r2 = r2_score(actual, predicted)

    mse = float(np.mean(errors ** 2))
    coverage = ((group["actual"] >= group["q10"]) & (group["actual"] <= group["q90"])).mean() * 100
    rows.append(
        {
            "target_name": target_name,
            "raw_actual_points": int(len(group)),
            "mae": float(np.mean(np.abs(errors))),
            "mse": mse,
            "rmse": float(np.sqrt(mse)),
            "bias": float(errors.mean()),
            "mape_percent": mape,
            "r2": r2,
            "q10_q90_coverage_percent": float(coverage),
        }
    )

metrics_df = pd.DataFrame(rows)
display(metrics_df.round(4))

## Runtime Summary

In [ ]:
runtime_summary = {
    "model_id": MODEL_ID,
    "device_map": DEVICE_MAP,
    "context_hours": CONTEXT_HOURS,
    "prediction_length": PREDICTION_LENGTH,
    "targets_count": len(targets),
    "prediction_rows": len(predictions_df),
    "comparison_rows_raw_actual_only": len(comparison_df),
    "model_load_seconds": load_seconds,
    "inference_seconds": inference_seconds,
    "inference_ms_per_prediction_row": inference_seconds * 1000 / len(predictions_df),
    "inference_ms_per_target": inference_seconds * 1000 / len(targets),
}
display(pd.DataFrame([runtime_summary]).round(4))